### Setup

These cells are copied unchanged from Section 1 of `exam2.ipynb`: the imports, the Black–Scholes functions and sample data of §1.4, and the simulation code of §1.6–§1.8. Run them before the Results below.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('MonteCarlo_PathDependency_HedgeControls')


#### 2.5.2 A daily-monitored barrier option

We now change the contract while keeping the Black–Scholes stock model. The up-and-out call introduces
path dependence and a discontinuity at the barrier, while retaining independent pricing benchmarks
(Lecture 5, pp.31–34). These let us examine both simulation error and the usefulness of the hedge-based
control introduced in §1.11.

We use the sample data, $S_0=K=100$, $T-t=1$, $r=5\%$ and $\sigma=20\%$, with the barrier $H=150$ from
the lecture's example (p.17). The option pays $\max(S_T-K,0)$ only if the stock closes below $H$ on all
252 observation dates, including expiry. Otherwise it pays nothing. Between observation dates its value
satisfies the Black–Scholes equation; on an observation date, its value is set to zero wherever $S\geq H$.
The continuously monitored version instead knocks out at the first crossing, including crossings between
daily observations. We keep these two contracts distinct throughout the comparison.


**Three benchmarks, none of them from Monte Carlo.**

1. *Continuous monitoring, closed form*, $V_{\text{cont}}$ (Rubinstein and Reiner, 1991). If the barrier were watched all the time, the method of images gives the
   price exactly: $V_{\text{cont}}(S,t) = c(S,t) - (S/H)^{1-2r/\sigma^2}\,c(H^2/S,\,t)$, where $c$ pays
   $\max(S_T-K,\,0)$ only if $S_T < H$. The two terms cancel at $S = H$.
2. *Daily monitoring, continuity correction* (Broadie, Glasserman and Kou, 1997; BGK). A path can cross $H$ between
   two closes and come back unpunished. So the daily contract behaves like a continuous one with a slightly higher
   barrier, $V_{\text{disc}} \approx V_{\text{cont}}\big(H e^{\beta\sigma\sqrt{\delta t}}\big)$, with
   $\beta = -\zeta(\tfrac12)/\sqrt{2\pi} \approx 0.5826$ ($\zeta$ is the Riemann zeta function).
3. *Daily monitoring, essentially exact*, $V_{\text{disc}}$. We work backwards through the 252 dates. At each date the
   value is the discounted integral of §1.3 over one day, taken only over prices below the barrier. In the log-price
   $x = \ln S$,
   $$
   V(x, t_i) = e^{-r\delta t}\int_{-\infty}^{\ln H} \tilde p\,(x, t_i;\, x', t_{i+1})\, V(x', t_{i+1})\, dx'.
   $$
   Each integral uses the midpoint rule on a fine grid whose top edge is exactly $\ln H$.

**The martingale control** (Wilmott, 2007, §29.13). The idea is simple. Alongside each path, we simulate the profit
and loss of a delta hedge. A hedge financed at the risk-free rate makes nothing on average under $\mathbb{Q}$, so its
P&L has zero mean. Subtracting it from the payoff therefore removes noise without adding bias. If the hedge is good,
the payoff and the hedge move together, and most of the noise goes.

For a hedge ratio $f(S, t)$, the discounted hedge P&L on one path, $Y$, and the controlled estimator are

$$
Y = \sum_{i=0}^{n-1} f(S_i, t_i)\; e^{-r t_{i+1}}\Big(\delta S_i - \mathbb E^{\mathbb Q}\big[\delta S_i \,\big|\, S_i\big]\Big),
\qquad
\mathbb E^{\mathbb Q}\big[\delta S_i \,\big|\, S_i\big] = S_i\big(e^{r\delta t} - 1\big),
$$

$$
\hat V_N = \frac1N\sum_{j=1}^N \big(P_j - \alpha\, Y_j\big),
\qquad
\alpha^* = \frac{\mathrm{Cov}[P,Y]}{\mathrm{Var}[Y]},
\qquad
\mathrm{Var}\big[P - \alpha^* Y\big] = (1-\rho_{PY}^2)\,\mathrm{Var}[P],
$$

where $P$ is the discounted payoff. Each term of $Y$ has zero mean as long as $f$ uses only what is known at $t_i$, so
the estimator is unbiased for any constant $\alpha$. We estimate $\alpha^*$ by regression on the same paths, so exact finite-sample unbiasedness is no longer guaranteed; the usual bias is of order $1/N$
under suitable regularity conditions. Wilmott discounts the whole sum once; we discount each increment from its own date,
so $Y$ is exactly the discounted P&L of the hedge. With $f = \Delta$, the replication argument of §1.2 says
$P \approx V_0 + Y$, so we expect $\alpha^* \approx 1$: a *delta-based control variate*.
Once a path is knocked out we set $f = 0$.

We try three hedge ratios: the Black–Scholes delta of the vanilla call (`bs_greeks`, §1.4), which is the Black–Scholes
model's own hedge; the delta of $V_{\text{cont}}$; and the delta of the BGK-corrected price.


**What we measure, and against what.**

| Label | What we compare | Question or expected behaviour |
|---|---|---|
| Table 2.5.2a | The quadrature with no barrier against `bs_price`, as its grid is refined; this also fixes $V_{\text{disc}}$ | It reproduces `bs_price` |
| Table 2.5.2b | The reference at 252, 1,008 and 4,032 monitoring dates against BGK and $V_{\text{cont}}$ | $V_{\text{disc}} > V_{\text{cont}}$, with the BGK correction suggesting a gap of order $\sqrt{\delta t}$ |
| Table 2.5.2c | The vanilla call: plain against the martingale control with the Black–Scholes delta | A large variance reduction, with $\alpha^* \approx 1$ |
| Table 2.5.2d, Figure 2.5.2a | The barrier: seven estimators at $N = 10{,}000$ on the same exact daily paths; then plain against the best control over 20 seeds | Does reducing sampling noise resolve the difference between the two contracts? |
| Figure 2.5.2b | The deltas against $S$, and the VRF of each control against the barrier level $H$ | The vanilla delta is a poor hedge near the barrier; as $H \to \infty$ every control recovers Table 2.5.2c |
| Table 2.5.2e | Rebalancing the hedge less often | For the call, the VRF is proportional to the number of rebalancing dates |
| Table 2.5.2f | Euler, Milstein and exact paths on the same draws | Do path errors change which paths knock out, and how does this affect price? |

The seven estimators are: plain (§1.6); antithetic (§1.9); a classical control variate, the vanilla call payoff,
whose mean is `bs_price`; the martingale control with each of the three deltas; and antithetic combined with the best
martingale control. The table reports price, SE, error against $V_{\text{disc}}$, the variance-reduction
factor $\mathrm{VRF} = \mathrm{SE}^2_{\text{plain}}/\mathrm{SE}^2$, run time, and the efficiency at equal run time,
$\mathrm{VRF}\times t_{\text{plain}}/t$. Figure 2.5.2a displays the confidence intervals. At this $N$ the gap $V_{\text{disc}} - V_{\text{cont}}$ is about 1.5 plain
standard errors: the effect sits "on the margins of the confidence interval".


**Benchmarks.** Table 2.5.2a checks the quadrature reference and fixes $V_{\text{disc}}$. Table 2.5.2b then links it,
through ever finer monitoring, to the BGK approximation and the continuous closed form.


**The martingale control.** The functions below compute three things on each path: the discounted barrier payoff,
the hedge P&L $Y$ for any hedge ratio $f(S, \tau)$ with $\tau = T - t$, and the controlled estimate with its standard
error. We try the control on the vanilla call first (Table 2.5.2c), then on the barrier (Table 2.5.2d).


**The cost of the hedge.** The barrier delta is expensive. It needs nine evaluations of the normal distribution
function $N(\cdot)$ on every path at every date, against two for the vanilla delta and none for plain sampling. A
cheaper route is to rebalance less often. Table 2.5.2e rebalances every 1, 2, 5, 10 or 21 trading days and holds the
position in between. $Y$ is still a martingale, because each holding is set using only what is known at its
rebalancing date.

For the vanilla call we can predict the result. Over a rebalancing interval $h$, the hedge leaves the gamma term of
Itô's lemma uncovered (§1.2), about $\tfrac12\Gamma S^2\sigma^2(\phi^2-1)\,h$. Its variance is of order $h^2$,
and there are $T/h$ intervals, so the variance of the hedging error is proportional to $h$. The VRF should therefore be
proportional to the number of rebalancing dates.


**Path accuracy.** Table 2.5.2f prices the barrier and the vanilla call with the three schemes of §1.8, on the same
50,000 sets of 252 draws (the first 10,000 are those used above). It also counts the paths whose knock-out status
differs from the exact path.


**Reading the results.** The daily benchmark is 7.7875, compared with 7.6224 for continuous
monitoring. Their difference is small enough for the plain Monte Carlo interval to contain both at
$N=10{,}000$. The BGK-delta control reduces variance by a factor of 46.3 and resolves the two prices
(Table 2.5.2d, Figure 2.5.2a). The repeated runs show that this improvement is not confined to the first seed.
Figure 2.5.2b connects it to the hedge: the barrier's exposure near knockout differs substantially from the
vanilla call's exposure.

The gain comes with computational cost. In the reported timings, daily barrier-delta evaluation gives no
clear advantage over antithetic sampling at equal runtime. Weekly rebalancing retains a variance-reduction
factor of 16.9 while reducing hedge evaluations (Table 2.5.2e). This is why both precision per path and
efficiency per unit time are reported.

Finally, Euler changes the knockout status of 39 of the 50,000 shared paths relative to exact simulation;
Milstein changes none (Table 2.5.2f). The discontinuous payoff makes path accuracy relevant, although one
sample cannot establish that a scheme will always make the same knockout decisions as the exact paths.
